# Urban heat and mobility patterns in NYC
# I-GUIDE Summer School 2026

**Team 5:** <br>
**Team Members:** Abhishek Anand, Josh Lown, Mirza Md Tasnim Mukarram, Sheida Ghahremani, Yuan Niu, Yuxin (Mia) Cao <br>
**Team Leader:** Yuqin Jiang

## 1. Project overview

This reproducible workshop compares predictive models for four
summer mobility outcomes across 1,025 NYC 1 km grid cells.
Predictors summarize environmental conditions, demographic
composition, and facility availability.

The analysis is predictive, not causal. Its output is not a final
drinking-fountain siting decision and does not show that a feature
causes mobility.

Dataset information:
[I-GUIDE dataset page](https://platform.i-guide.io/datasets/dd6f25ee-d734-45d4-8487-c3d804684afb).
This page is documentation, not a CSV download link.

## 2. Imports and reproducibility

The notebook never installs or upgrades packages. XGBoost is a
required model; if unavailable, the import cell provides a clear
setup error. All randomized operations use seed 42. One worker is
used to avoid oversubscribing a shared JupyterHub.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import platform
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "This workshop includes XGBoost. Install the xgboost package "
        "in the JupyterHub kernel, then restart and rerun."
    ) from exc

RANDOM_STATE = 42
N_JOBS = 1
CV_SPLITS = 5
NOTEBOOK_START = time.perf_counter()
np.random.seed(RANDOM_STATE)

def concise_warning(message, category, filename, lineno, file=None, line=None):
    stream = file if file is not None else sys.stderr
    print(f"{category.__name__}: {message}", file=stream)


warnings.showwarning = concise_warning
warnings.filterwarnings("once", category=ConvergenceWarning)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.2,
        "font.size": 10,
    }
)

package_versions = pd.DataFrame(
    {
        "package": [
            "Python",
            "pandas",
            "numpy",
            "matplotlib",
            "scikit-learn",
            "xgboost",
        ],
        "version": [
            platform.python_version(),
            metadata.version("pandas"),
            metadata.version("numpy"),
            metadata.version("matplotlib"),
            metadata.version("scikit-learn"),
            metadata.version("xgboost"),
        ],
    }
)
display(package_versions)

## 3. Load and validate data

Place the I-GUIDE `all.csv` file in the same folder as this
notebook. Shape, IDs, missingness, and file hash are computed at
run time.

In [ ]:
DATASET_PAGE_URL = (
    "https://platform.i-guide.io/datasets/"
    "dd6f25ee-d734-45d4-8487-c3d804684afb"
)

DATA_PATH = Path("all.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "all.csv was not found. Attach or download the I-GUIDE dataset "
        "into the same folder as this notebook. Dataset page: "
        f"{DATASET_PAGE_URL}"
    )

df = pd.read_csv(DATA_PATH)
data_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
data_overview = pd.DataFrame(
    [
        {
            "rows": len(df),
            "columns": len(df.columns),
            "sha256": data_sha256,
            "duplicate_uid_num": (
                int(df["uid_num"].duplicated().sum())
                if "uid_num" in df.columns else np.nan
            ),
            "duplicate_grid_id": (
                int(df["grid_id"].duplicated().sum())
                if "grid_id" in df.columns else np.nan
            ),
            "missing_cells_raw": int(df.isna().sum().sum()),
        }
    ]
)
display(data_overview)
print("Columns:")
print(df.columns.tolist())

## 4. Variable definitions

The final predictor set has exactly 16 variables and uses
`pct_u5` rather than `pct_u18`. The source field `internal_t` and
every mobility outcome are excluded from predictors. Although
`pct_white` and `pct_nonwhite` are redundant, both remain in the
agreed team feature set and are noted as a limitation.

Positive net flow means net attraction: inflow minus outflow.

In [ ]:
FEATURES = [
    "total_faci",
    "total_DF",
    "NDVI",
    "GHSLfrac",
    "total_SS",
    "NDWI_Media",
    "LST_summer",
    "HHInc_Med_weighted",
    "pct_u5",
    "pct_65plus",
    "pct_white",
    "pct_black",
    "pct_asian",
    "pct_other_race",
    "pct_two_or_more",
    "pct_nonwhite",
]
OUTCOMES = {
    "Inflow": "inflow_jja",
    "Outflow": "outflow_jja",
    "Internal flow": "internal_trips_jja",
    "Net flow": "net_flow_jja",
}
SOURCE_OUTCOMES = [
    "inflow_jja",
    "outflow_jja",
    "internal_trips_jja",
]
ID_COLUMNS = ["uid_num", "grid_id"]

assert len(FEATURES) == 16
assert FEATURES[8] == "pct_u5"

required_columns = set(
    ID_COLUMNS
    + FEATURES
    + SOURCE_OUTCOMES
    + ["outdoor_DF", "indoor_DF"]
)
missing_required = sorted(required_columns.difference(df.columns))
if missing_required:
    raise KeyError(f"Required columns are missing: {missing_required}")

variable_table = pd.DataFrame(
    {
        "role": ["predictor"] * len(FEATURES)
        + ["outcome"] * len(OUTCOMES),
        "variable": FEATURES + list(OUTCOMES.values()),
    }
)
display(variable_table)

## 5. Data preprocessing

Infinite numeric values become missing. Blank `total_DF` is set to
zero only when both outdoor and indoor fountain counts are zero,
which is the confirmed structural-zero rule. Other predictor
missingness stays in the data.

Every model pipeline fits median imputation with missingness
indicators and standardization on training data only. Test data
are transform-only.

In [ ]:
numeric_columns = list(
    dict.fromkeys(
        FEATURES
        + SOURCE_OUTCOMES
        + ["outdoor_DF", "indoor_DF"]
    )
)
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

infinity_before = pd.Series(
    {
        column: int(np.isinf(df[column]).sum())
        for column in numeric_columns
    },
    name="infinite_count",
)
df[numeric_columns] = df[numeric_columns].replace(
    [np.inf, -np.inf], np.nan
)

total_df_missing = df["total_DF"].isna()
components_zero = (
    df.loc[total_df_missing, "outdoor_DF"].eq(0)
    & df.loc[total_df_missing, "indoor_DF"].eq(0)
)
if total_df_missing.any() and not components_zero.all():
    raise ValueError(
        "The total_DF structural-zero rule does not hold for every "
        "missing total_DF row."
    )
df.loc[total_df_missing, "total_DF"] = 0
df["net_flow_jja"] = df["inflow_jja"] - df["outflow_jja"]

if df[SOURCE_OUTCOMES].isna().any().any():
    raise ValueError("Mobility outcomes contain missing values.")
if (df[SOURCE_OUTCOMES] < 0).any().any():
    raise ValueError("A nonnegative mobility outcome contains negatives.")

X = df.loc[:, FEATURES].copy()
assert list(X.columns) == FEATURES
preprocessing_qc = pd.DataFrame(
    {
        "variable": FEATURES,
        "missing_after_rules": [
            int(X[column].isna().sum()) for column in FEATURES
        ],
        "infinity_before_replacement": [
            int(infinity_before.get(column, 0)) for column in FEATURES
        ],
    }
)
print(
    "Structural-zero replacements for total_DF:",
    int(total_df_missing.sum()),
)
display(preprocessing_qc)

## 6. Shared train-test split

Every model and outcome uses the same grids. A valid split column
in an uploaded dataset is honored. Otherwise the audited team split
is recreated by stratifying `log1p(inflow)` quantile bins with
seed 42. For the audited data this exactly matches the existing
820/205 split.

In [ ]:
df = df.sort_values("uid_num").reset_index(drop=True)
X = df.loc[:, FEATURES].copy()

if "split" in df.columns:
    split_labels = df["split"].astype(str).str.lower()
    if not split_labels.isin(["train", "test"]).all():
        raise ValueError("The split column must contain train/test only.")
    split_source = "all.csv split column"
else:
    stratification_bins = pd.qcut(
        np.log1p(df["inflow_jja"]),
        q=10,
        labels=False,
        duplicates="drop",
    )
    train_index, test_index = train_test_split(
        df.index,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=stratification_bins,
    )
    split_labels = pd.Series("train", index=df.index, dtype="object")
    split_labels.loc[test_index] = "test"
    split_source = (
        "deterministic log1p(inflow) quantile-stratified split"
    )

df["split"] = split_labels.to_numpy()
train_mask = df["split"].eq("train")
test_mask = df["split"].eq("test")
train_uids = set(df.loc[train_mask, "uid_num"])
test_uids = set(df.loc[test_mask, "uid_num"])
split_qc = pd.DataFrame(
    [
        {
            "source": split_source,
            "train_count": int(train_mask.sum()),
            "test_count": int(test_mask.sum()),
            "uid_overlap": len(train_uids.intersection(test_uids)),
            "duplicate_uid_num": int(df["uid_num"].duplicated().sum()),
            "duplicate_grid_id": int(df["grid_id"].duplicated().sum()),
        }
    ]
)
assert int(train_mask.sum()) == 820
assert int(test_mask.sum()) == 205
assert train_uids.isdisjoint(test_uids)
assert not df["uid_num"].duplicated().any()
assert not df["grid_id"].duplicated().any()

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
assert list(X_train.columns) == FEATURES
assert list(X_test.columns) == FEATURES
display(split_qc)

## 7. Exploratory analysis

These concise diagnostics show coverage, outcome distributions,
and correlation structure. Correlation is descriptive, not causal.

In [ ]:
analysis_columns = FEATURES + list(OUTCOMES.values())
display(df[analysis_columns].describe().T.round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, (outcome_name, column) in zip(axes.flat, OUTCOMES.items()):
    axis.hist(df[column], bins=35, color="#4C78A8", alpha=0.85)
    axis.set_title(outcome_name)
    axis.set_xlabel("Trips")
    axis.set_ylabel("Grid cells")
fig.suptitle("Summer mobility outcome distributions", y=1.02)
fig.tight_layout()
plt.show()
plt.close(fig)

correlation = df[analysis_columns].corr()
fig, axis = plt.subplots(figsize=(14, 12))
image = axis.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
axis.set_xticks(range(len(correlation.columns)), correlation.columns)
axis.set_yticks(range(len(correlation.index)), correlation.index)
axis.tick_params(axis="x", rotation=90)
axis.set_title("Predictor and outcome correlation matrix")
fig.colorbar(image, ax=axis, label="Pearson correlation")
fig.tight_layout()
plt.show()
plt.close(fig)

## 8. Models and training-only tuning

Four families are retrained: Random Forest, RBF-SVR, a
scikit-learn MLP ANN adaptation, and XGBoost. The MLP retains the
team's 64–32–16 hidden widths, training-only transformed-target
standardization, and early stopping, but is not presented as an
exact reproduction of the Colab/PyTorch model.

Compact hyperparameter grids keep the workshop practical. Every
search uses five shuffled training folds; the test set remains
untouched until final evaluation.

In [ ]:
def signed_log1p(values):
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.log1p(np.abs(values))


def signed_expm1(values):
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.expm1(np.abs(values))


def target_functions(outcome_name):
    if outcome_name == "Net flow":
        return signed_log1p, signed_expm1, "signed_log1p"
    return np.log1p, np.expm1, "log1p"


def make_estimator(model_name, outcome_name):
    if model_name == "Random Forest":
        model = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
    elif model_name == "RBF-SVR":
        model = SVR(kernel="rbf", cache_size=1000)
    elif model_name == "ANN (sklearn MLP)":
        model = MLPRegressor(
            hidden_layer_sizes=(64, 32, 16),
            activation="relu",
            solver="adam",
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=40,
            max_iter=1000,
            random_state=RANDOM_STATE,
        )
    elif model_name == "XGBoost":
        model = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbosity=0,
        )
    else:
        raise KeyError(model_name)

    pipeline = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="median", add_indicator=True),
            ),
            ("scaler", StandardScaler()),
            ("model", model),
        ]
    )
    function, inverse_function, _ = target_functions(outcome_name)
    if model_name == "ANN (sklearn MLP)":
        target_transformer = Pipeline(
            [
                (
                    "nonlinear",
                    FunctionTransformer(
                        function,
                        inverse_func=inverse_function,
                        validate=False,
                        check_inverse=False,
                    ),
                ),
                ("scaler", StandardScaler()),
            ]
        )
        return TransformedTargetRegressor(
            regressor=pipeline,
            transformer=target_transformer,
            check_inverse=False,
        )
    return TransformedTargetRegressor(
        regressor=pipeline,
        func=function,
        inverse_func=inverse_function,
        check_inverse=False,
    )


PARAMETER_GRIDS = {
    "Random Forest": {
        "regressor__model__n_estimators": [300],
        "regressor__model__max_features": ["sqrt", 0.75],
        "regressor__model__min_samples_leaf": [2, 5],
        "regressor__model__max_depth": [None, 20],
    },
    "RBF-SVR": {
        "regressor__model__C": [1, 10, 100],
        "regressor__model__gamma": [0.01, 0.1, "scale"],
        "regressor__model__epsilon": [0.05, 0.1],
    },
    "ANN (sklearn MLP)": {
        "regressor__model__activation": ["relu"],
        "regressor__model__alpha": [0.01, 0.1, 1.0, 10.0],
        "regressor__model__learning_rate_init": [0.0001, 0.0003],
    },
    "XGBoost": {
        "regressor__model__n_estimators": [350, 700],
        "regressor__model__max_depth": [3, 6],
        "regressor__model__learning_rate": [0.02, 0.05],
        "regressor__model__min_child_weight": [3],
        "regressor__model__subsample": [0.75],
        "regressor__model__colsample_bytree": [0.75],
        "regressor__model__reg_lambda": [10.0],
        "regressor__model__gamma": [0.1],
    },
}
MODEL_ORDER = list(PARAMETER_GRIDS)
OUTCOME_ORDER = list(OUTCOMES)
cv = KFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
print("Models:", MODEL_ORDER)
print("Outcomes:", OUTCOME_ORDER)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 9. Mobility outcomes and model fitting

Inflow, outflow, and internal trips use `log1p`. Net flow uses a
signed-log transform. Predictions are inverse-transformed before
evaluation. Negative predictions for nonnegative outcomes are
clipped to zero; net-flow predictions are not clipped.

In [ ]:
metric_rows = []
prediction_frames = []
fitted_models = {}

for model_name in MODEL_ORDER:
    for outcome_name, target_column in OUTCOMES.items():
        run_start = time.perf_counter()
        y = df[target_column].astype(float)
        y_train = y.loc[train_mask]
        y_test = y.loc[test_mask]

        search = GridSearchCV(
            estimator=make_estimator(model_name, outcome_name),
            param_grid=PARAMETER_GRIDS[model_name],
            scoring="r2",
            cv=cv,
            n_jobs=N_JOBS,
            refit=True,
            error_score="raise",
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_

        cv_r2, cv_rmse, cv_mae = [], [], []
        for fold_train, fold_validation in cv.split(X_train):
            fold_model = clone(best_model)
            fold_model.fit(
                X_train.iloc[fold_train],
                y_train.iloc[fold_train],
            )
            fold_prediction = fold_model.predict(
                X_train.iloc[fold_validation]
            )
            if outcome_name != "Net flow":
                fold_prediction = np.clip(fold_prediction, 0, None)
            fold_observed = y_train.iloc[fold_validation].to_numpy()
            cv_r2.append(r2_score(fold_observed, fold_prediction))
            cv_rmse.append(
                np.sqrt(
                    mean_squared_error(
                        fold_observed, fold_prediction
                    )
                )
            )
            cv_mae.append(
                mean_absolute_error(
                    fold_observed, fold_prediction
                )
            )

        predicted_raw = best_model.predict(X_test)
        if outcome_name == "Net flow":
            predicted = predicted_raw.copy()
            was_clipped = np.zeros(len(predicted), dtype=bool)
        else:
            predicted = np.clip(predicted_raw, 0, None)
            was_clipped = predicted_raw < 0

        observed = y_test.to_numpy()
        mse = mean_squared_error(observed, predicted)
        _, _, transform_name = target_functions(outcome_name)
        elapsed = time.perf_counter() - run_start
        metric_rows.append(
            {
                "model": model_name,
                "outcome": outcome_name,
                "target_column": target_column,
                "transformation": transform_name,
                "n_train": len(X_train),
                "n_test": len(X_test),
                "cv_r2_mean": float(np.mean(cv_r2)),
                "cv_r2_std": float(np.std(cv_r2, ddof=1)),
                "cv_rmse_mean": float(np.mean(cv_rmse)),
                "cv_mae_mean": float(np.mean(cv_mae)),
                "test_r2": float(r2_score(observed, predicted)),
                "test_rmse": float(np.sqrt(mse)),
                "test_mae": float(
                    mean_absolute_error(observed, predicted)
                ),
                "negative_prediction_count": int(
                    (predicted_raw < 0).sum()
                ),
                "runtime_seconds": elapsed,
                "best_parameters": json.dumps(
                    search.best_params_, sort_keys=True
                ),
            }
        )
        prediction_frames.append(
            pd.DataFrame(
                {
                    "uid_num": df.loc[test_mask, "uid_num"].to_numpy(),
                    "grid_id": df.loc[test_mask, "grid_id"].to_numpy(),
                    "model": model_name,
                    "outcome": outcome_name,
                    "observed": observed,
                    "predicted_raw": predicted_raw,
                    "predicted": predicted,
                    "was_clipped": was_clipped,
                    "residual": observed - predicted,
                }
            )
        )
        fitted_models[(model_name, outcome_name)] = best_model
        print(
            f"Completed {model_name} | {outcome_name}: "
            f"test R²={metric_rows[-1]['test_r2']:.3f}, "
            f"{elapsed:.1f}s"
        )

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)
assert len(metrics) == len(MODEL_ORDER) * len(OUTCOME_ORDER)
assert metrics[["model", "outcome"]].duplicated().sum() == 0
assert metrics["n_train"].eq(820).all()
assert metrics["n_test"].eq(205).all()
assert predictions.groupby(["model", "outcome"]).size().eq(205).all()
assert predictions.loc[
    predictions["outcome"].ne("Net flow"), "predicted"
].ge(0).all()
assert not predictions.loc[
    predictions["outcome"].eq("Net flow"), "was_clipped"
].any()

display(
    metrics[
        [
            "model",
            "outcome",
            "transformation",
            "cv_r2_mean",
            "cv_r2_std",
            "cv_rmse_mean",
            "test_r2",
            "test_rmse",
            "test_mae",
            "negative_prediction_count",
            "runtime_seconds",
        ]
    ]
    .sort_values(["outcome", "test_r2"], ascending=[True, False])
    .round(4)
)

Completed ANN (sklearn MLP) | Net flow: test R²=-0.011, 31.1s


Completed XGBoost | Inflow: test R²=0.716, 15.5s


Completed XGBoost | Outflow: test R²=0.612, 15.9s


Completed XGBoost | Internal flow: test R²=0.588, 16.3s


Completed XGBoost | Net flow: test R²=0.077, 20.0s


,model,outcome,transformation,cv_r2_mean,cv_r2_std,cv_rmse_mean,test_r2,test_rmse,test_mae,negative_prediction_count,runtime_seconds
12,XGBoost,Inflow,log1p,0.5879,0.0810,62348.3764,0.7164,47759.5290,28888.6688,0,15.4738
0,Random Forest,Inflow,log1p,0.5696,0.0600,63847.4958,0.6462,53346.6372,32266.8461,0,35.1988
4,RBF-SVR,Inflow,log1p,0.6089,0.1031,60509.4616,0.4916,63943.7144,35352.6215,0,6.3754
8,ANN (sklearn MLP),Inflow,log1p,0.3475,0.0779,78527.0436,0.2744,76391.2290,44242.3851,0,68.5073
14,XGBoost,Internal flow,log1p,0.4216,0.0537,8550.4593,0.5884,7030.3036,3933.2380,2,16.3212
2,Random Forest,Internal flow,log1p,0.3913,0.0497,8771.4987,0.5237,7563.5422,4178.3135,0,35.7723
6,RBF-SVR,Internal flow,log1p,0.4253,0.0678,8518.9493,0.3630,8746.1905,4809.3166,5,7.4090
10,ANN (sklearn MLP),Internal flow,log1p,0.1524,0.0532,10353.5959,0.1764,9945.1892,5413.0514,0,54.0824
7,RBF-SVR,Net flow,signed_log1p,0.0255,0.0172,63408.6632,0.1011,45663.9032,28475.9551,176,4.9571
15,XGBoost,Net flow,signed_log1p,0.0421,0.0943,62874.5658,0.0772,46266.9885,29092.0729,140,19.9970


## 10. Model evaluation

Main metrics are computed after inverse transformation on the
original trip scale. R² supports auxiliary comparison across
outcomes. RMSE and MAE should compare models only within the same
outcome because the four outcome scales differ. Percentage error
is omitted because outcomes contain zeros and net flow is signed.

In [ ]:
best_by_outcome = (
    metrics.loc[
        metrics.groupby("outcome", sort=False)["test_r2"].idxmax(),
        [
            "outcome",
            "model",
            "test_r2",
            "test_rmse",
            "test_mae",
            "cv_r2_mean",
            "cv_r2_std",
        ],
    ]
    .set_index("outcome")
    .loc[OUTCOME_ORDER]
    .reset_index()
)
display(best_by_outcome.round(4))
display(
    metrics[["model", "outcome", "best_parameters"]]
    .sort_values(["model", "outcome"])
)

## 11. Visualizations

Every figure is generated from this run; no presentation
screenshot or pre-rendered local image is used.

In [ ]:
r2_table = (
    metrics.pivot(index="outcome", columns="model", values="test_r2")
    .loc[OUTCOME_ORDER, MODEL_ORDER]
)
axis = r2_table.plot.bar(figsize=(13, 6), width=0.8)
axis.axhline(0, color="black", linewidth=0.8)
axis.set_title("Test R² by model and mobility outcome")
axis.set_xlabel("Outcome")
axis.set_ylabel("Test R²")
axis.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for axis, outcome_name in zip(axes.flat, OUTCOME_ORDER):
    subset = (
        metrics.loc[metrics["outcome"].eq(outcome_name)]
        .set_index("model")
        .loc[MODEL_ORDER]
    )
    axis.bar(subset.index, subset["test_rmse"], color="#F58518")
    axis.set_title(outcome_name)
    axis.set_ylabel("Test RMSE (original trips)")
    axis.tick_params(axis="x", rotation=30)
fig.suptitle(
    "Test RMSE by outcome (compare bars only within each panel)",
    y=1.02,
)
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(
    len(MODEL_ORDER), len(OUTCOME_ORDER), figsize=(16, 15)
)
for row, model_name in enumerate(MODEL_ORDER):
    for column_index, outcome_name in enumerate(OUTCOME_ORDER):
        axis = axes[row, column_index]
        subset = predictions.loc[
            predictions["model"].eq(model_name)
            & predictions["outcome"].eq(outcome_name)
        ]
        lower = min(
            subset["observed"].min(), subset["predicted"].min()
        )
        upper = max(
            subset["observed"].max(), subset["predicted"].max()
        )
        axis.scatter(
            subset["observed"],
            subset["predicted"],
            alpha=0.55,
            s=16,
        )
        axis.plot([lower, upper], [lower, upper], "k--", linewidth=1)
        score = metrics.loc[
            metrics["model"].eq(model_name)
            & metrics["outcome"].eq(outcome_name)
        ].iloc[0]
        axis.set_title(
            f"{model_name}\n{outcome_name}\n"
            f"R²={score['test_r2']:.3f}, "
            f"RMSE={score['test_rmse']:.0f}"
        )
        axis.set_xlabel("Observed")
        axis.set_ylabel("Predicted")
fig.suptitle(
    "Observed versus predicted values on shared test grids", y=1.01
)
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(
    len(MODEL_ORDER), len(OUTCOME_ORDER), figsize=(16, 15)
)
for row, model_name in enumerate(MODEL_ORDER):
    for column_index, outcome_name in enumerate(OUTCOME_ORDER):
        axis = axes[row, column_index]
        subset = predictions.loc[
            predictions["model"].eq(model_name)
            & predictions["outcome"].eq(outcome_name)
        ]
        axis.scatter(
            subset["predicted"],
            subset["residual"],
            alpha=0.55,
            s=16,
            color="#54A24B",
        )
        axis.axhline(0, color="black", linestyle="--", linewidth=1)
        axis.set_title(f"{model_name}\n{outcome_name}")
        axis.set_xlabel("Predicted")
        axis.set_ylabel("Observed − predicted")
fig.suptitle("Residual diagnostics on shared test grids", y=1.01)
fig.tight_layout()
plt.show()
plt.close(fig)

### Permutation feature importance

Importance is the change in test R² after shuffling one feature.
Positive values mean shuffling harms prediction. Negative values
can occur when shuffling slightly improves a finite test score;
they suggest noise, redundancy, correlation, or instability—not a
beneficial causal effect.

In [ ]:
importance_rows = []
for model_name in MODEL_ORDER:
    for outcome_name, target_column in OUTCOMES.items():
        estimator = fitted_models[(model_name, outcome_name)]
        y_test = df.loc[test_mask, target_column].to_numpy()

        def clipped_r2(model, features, observed):
            values = model.predict(features)
            if outcome_name != "Net flow":
                values = np.clip(values, 0, None)
            return r2_score(observed, values)

        result = permutation_importance(
            estimator,
            X_test,
            y_test,
            scoring=clipped_r2,
            n_repeats=8,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
        for feature, mean_value, sd_value in zip(
            FEATURES,
            result.importances_mean,
            result.importances_std,
        ):
            importance_rows.append(
                {
                    "model": model_name,
                    "outcome": outcome_name,
                    "feature": feature,
                    "importance_mean": float(mean_value),
                    "importance_std": float(sd_value),
                }
            )

importance = pd.DataFrame(importance_rows)
assert len(importance) == (
    len(MODEL_ORDER) * len(OUTCOME_ORDER) * len(FEATURES)
)
combined_importance = (
    importance.groupby(["model", "feature"], as_index=False)
    ["importance_mean"]
    .mean()
)
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
for axis, model_name in zip(axes.flat, MODEL_ORDER):
    subset = (
        combined_importance.loc[
            combined_importance["model"].eq(model_name)
        ]
        .sort_values("importance_mean")
    )
    colors = np.where(
        subset["importance_mean"].ge(0), "#4C78A8", "#E45756"
    )
    axis.barh(
        subset["feature"], subset["importance_mean"], color=colors
    )
    axis.axvline(0, color="black", linewidth=0.8)
    axis.set_title(model_name)
    axis.set_xlabel("Mean test-R² decrease across outcomes")
fig.suptitle(
    "Permutation feature importance combined across targets",
    y=1.01,
)
fig.tight_layout()
plt.show()
plt.close(fig)

svm_importance = (
    importance.loc[importance["model"].eq("RBF-SVR")]
    .pivot(
        index="feature",
        columns="outcome",
        values="importance_mean",
    )
    .loc[:, OUTCOME_ORDER]
)
svm_order = svm_importance.mean(axis=1).sort_values().index
axis = svm_importance.loc[svm_order].plot.barh(
    figsize=(13, 9), width=0.8
)
axis.axvline(0, color="black", linewidth=0.8)
axis.set_title("RBF-SVR permutation importance across all targets")
axis.set_xlabel("Test-R² decrease after permutation")
axis.set_ylabel("Predictor")
axis.legend(title="Outcome")
plt.tight_layout()
plt.show()
plt.close()

## 12. Results and interpretation

Summaries are generated from current metrics, not hard-coded.
Importance describes model reliance and association, not causality.

In [ ]:
for row in best_by_outcome.itertuples(index=False):
    print(
        f"{row.outcome}: best test R² = {row.test_r2:.3f} "
        f"from {row.model}; RMSE = {row.test_rmse:,.1f}, "
        f"MAE = {row.test_mae:,.1f}."
    )

svm_results = (
    metrics.loc[metrics["model"].eq("RBF-SVR")]
    .set_index("outcome")
    .loc[OUTCOME_ORDER]
)
print(
    "\nRBF-SVR test R² range: "
    f"{svm_results['test_r2'].min():.3f} to "
    f"{svm_results['test_r2'].max():.3f}."
)
ann_results = (
    metrics.loc[metrics["model"].eq("ANN (sklearn MLP)")]
    .set_index("outcome")
    .loc[OUTCOME_ORDER]
)
print(
    "ANN adaptation test R² range: "
    f"{ann_results['test_r2'].min():.3f} to "
    f"{ann_results['test_r2'].max():.3f}. "
    "With 820 training grids, results can be sensitive to "
    "architecture, regularization, and tuning budget."
)
net_scores = metrics.loc[
    metrics["outcome"].eq("Net flow"), "test_r2"
]
other_scores = metrics.loc[
    metrics["outcome"].ne("Net flow"), "test_r2"
]
if net_scores.mean() < other_scores.mean():
    print(
        "Net flow is harder on average than the nonnegative flows "
        "under test R²."
    )
else:
    print(
        "Net flow is not hardest on average in this run, although "
        "its signed residual structure remains challenging."
    )

cross_model_importance = (
    importance.groupby("feature")["importance_mean"]
    .mean()
    .sort_values(ascending=False)
)
print(
    "Highest mean permutation importance across models/outcomes:"
)
display(cross_model_importance.head(8).rename("mean_importance"))
negative_importance_count = int(
    importance["importance_mean"].lt(0).sum()
)
print(
    f"Negative mean permutation importances: "
    f"{negative_importance_count} of {len(importance)}. "
    "Treat them as unstable or redundant contributions, not "
    "protective causal effects."
)

## 13. Limitations

- Only 1,025 spatial grids are available; complex models can
  overfit despite validation and regularization.
- A random split can overstate generalization to spatially distinct
  neighborhoods; spatial blocking is a useful sensitivity test.
- These models are predictive, not causal.
- Race and income variables require contextual, careful
  interpretation and are not intrinsic risk.
- Mobility is not drinking-fountain suitability.
- Correlated and compositional predictors, including the retained
  white/nonwhite pair, can destabilize importance.
- Tuning budgets are compact and not perfectly equal in effective
  complexity across model families.
- The scikit-learn MLP is a platform-compatible ANN adaptation, not
  an exact PyTorch reproduction.

## 14. Conclusion

This notebook gives one auditable comparison of four predictive
model families using the same predictors, transformed outcomes,
and shared test grids. It supports workshop discussion but does not
turn mobility predictions into a siting recommendation.

In [ ]:
total_runtime = time.perf_counter() - NOTEBOOK_START
final_qc = pd.DataFrame(
    [
        {
            "data_rows": len(df),
            "predictors": len(FEATURES),
            "outcomes": len(OUTCOMES),
            "models": len(MODEL_ORDER),
            "metric_rows": len(metrics),
            "train_rows": int(train_mask.sum()),
            "test_rows": int(test_mask.sum()),
            "uid_overlap": len(train_uids.intersection(test_uids)),
            "runtime_seconds": total_runtime,
        }
    ]
)
display(final_qc.round(3))
print("Workshop execution completed successfully.")